# Gás do Povo — DAG para Avaliação de Impacto

Este notebook constrói o Directed Acyclic Graph (DAG) metodológico para uma **futura avaliação de impacto** do programa Gás do Povo.

O DAG não descreve uma análise causal disponível com os dados atuais: os dados coletados nos notebooks anteriores (pagamentos por voucher e consolidado por município) **não contêm resultado social externo nem contrafactual**, portanto apenas estudos associativos são viáveis com as fontes atuais.

O DAG apresenta a estrutura causal proposta e indica quais dados adicionais são necessários para identificar o efeito causal.

In [ ]:
!pip install --upgrade matplotlib numpy pandas --quiet

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np

DAG_PATH = "dag_gas_do_povo.png"

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9), dpi=200)
fig.patch.set_facecolor("#F7F8FA")
ax.set_facecolor("#F7F8FA")
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis("off")

# ── Posições dos nós ──────────────────────────────────────────────────────────
pos = {
    "cad":   (2.5,  6.5),   # Elegibilidade / CadÚnico
    "gdp":   (8.0,  6.5),   # Gás do Povo (tratamento)
    "out":   (13.5, 6.5),   # Resultado social externo
    "mech":  (10.8, 3.8),   # Mecanismos pós-programa
    "dados": (5.2,  3.8),   # Dados adicionais necessários
}

# ── Especificações visuais de cada nó ────────────────────────────────────────
specs = {
    "cad":   ("Características prévias",
              "CadÚnico · renda · BPC · chefia feminina · região",
              "#DCEBFA", "#3977A8"),
    "gdp":   ("Gás do Povo",
              "tratamento: recebimento do voucher de gás",
              "#DDF3E4", "#378253"),
    "out":   ("Resultado social externo",
              "SAN · saúde · renda · combustível limpo",
              "#FADDDD", "#B84D4D"),
    "mech":  ("Mecanismos pós-programa",
              "acesso a gás · redução de biomassa · renda disponível",
              "#FFF0C9", "#B78320"),
    "dados": ("Dados adicionais necessários",
              "painel pré/pós · unidades comparáveis · outcome externo",
              "#EEE9FF", "#7663A8"),
}


def arrow(a, b, color="#586273", curve=0.0, dashed=False):
    ax.add_patch(FancyArrowPatch(
        pos[a], pos[b],
        arrowstyle="-|>", mutation_scale=16, linewidth=2.0,
        color=color, linestyle="dashed" if dashed else "solid",
        connectionstyle=f"arc3,rad={curve}",
        shrinkA=78, shrinkB=78, zorder=1,
    ))


# ── Arestas causais ───────────────────────────────────────────────────────────
arrow("cad",   "gdp",   color="#3977A8")                   # elegibilidade → tratamento
arrow("cad",   "out",   color="#3977A8", curve=-0.18)      # confundidor direto
arrow("gdp",   "out",   color="#378253")                   # efeito total do tratamento
arrow("gdp",   "mech",  color="#B78320")                   # tratamento → mecanismo
arrow("mech",  "out",   color="#B78320", curve=-0.12)      # mecanismo → resultado
arrow("dados", "gdp",   color="#7663A8", dashed=True)      # dado necessário (tracejado)
arrow("dados", "out",   color="#7663A8", curve=-0.15, dashed=True)

# ── Nós ───────────────────────────────────────────────────────────────────────
for key, (title, subtitle, fill, edge) in specs.items():
    x, y = pos[key]
    w = 4.0 if key != "dados" else 4.3
    ax.add_patch(FancyBboxPatch(
        (x - w / 2, y - 0.56), w, 1.12,
        boxstyle="round,pad=0.06,rounding_size=0.13",
        facecolor=fill, edgecolor=edge, linewidth=1.7, zorder=3,
    ))
    ax.text(x, y + 0.16, title,    ha="center", va="center",
            fontsize=11, fontweight="bold", color="#172033", zorder=4)
    ax.text(x, y - 0.21, subtitle, ha="center", va="center",
            fontsize=8.4, color="#4B5563", zorder=4)

# ── Título e subtítulo ────────────────────────────────────────────────────────
ax.text(0.5, 8.55, "DAG proposto para uma futura avaliação de impacto do Gás do Povo",
        fontsize=20, fontweight="bold", color="#172033")
ax.text(0.5, 8.12,
        "O resultado deve ser externo ao pagamento do voucher e mensurado antes e após o programa",
        fontsize=11.5, color="#536074")

# ── Nota metodológica ─────────────────────────────────────────────────────────
note = (
    "ANÁLISE DISPONÍVEL AGORA  •  As bases atuais (Portal da Transparência e dados.gov.br) "
    "permitem estudar\n"
    "fatores associados à cobertura e ao valor do benefício. Elas não contêm resultado social externo "
    "nem contrafactual,\n"
    "portanto não identificam impacto causal. Para avaliação de impacto, é necessário um painel "
    "município-ano com desfecho\n"
    "externo (ex.: segurança alimentar, saúde) e um desenho quasi-experimental (DiD, PSM, RDD)."
)
ax.text(0.5, 1.6, note, fontsize=9.8, color="#7A4D11", ha="left", va="center",
        bbox=dict(boxstyle="round,pad=0.6", facecolor="#FFF7E3",
                  edgecolor="#D7A94E", linewidth=1.3))

# ── Legenda ───────────────────────────────────────────────────────────────────
legend_items = [
    mpatches.Patch(facecolor="#DCEBFA", edgecolor="#3977A8", label="Características prévias (confundidores)"),
    mpatches.Patch(facecolor="#DDF3E4", edgecolor="#378253", label="Tratamento (Gás do Povo)"),
    mpatches.Patch(facecolor="#FADDDD", edgecolor="#B84D4D", label="Resultado social externo"),
    mpatches.Patch(facecolor="#FFF0C9", edgecolor="#B78320", label="Mecanismos pós-programa"),
    mpatches.Patch(facecolor="#EEE9FF", edgecolor="#7663A8", label="Dados adicionais necessários"),
]
ax.legend(handles=legend_items, loc="lower right", fontsize=9,
          framealpha=0.85, edgecolor="#CCCCCC",
          bbox_to_anchor=(0.99, 0.01))

plt.savefig(DAG_PATH, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"DAG salvo em: {DAG_PATH}")

## Interpretação do DAG

### Nós

| Nó | Papel | Exemplos de variáveis |
|----|-------|----------------------|
| **Características prévias** | Confundidores e determinantes da elegibilidade | Inscrição no CadÚnico, renda per capita ≤ ½ SM, recebimento de BPC, chefia feminina, região |
| **Gás do Povo** | Tratamento (variável de interesse) | Recebimento do voucher; valor; competência |
| **Resultado social externo** | Desfecho de interesse causal | Segurança Alimentar e Nutricional (SAN), gastos com energia, internações por doenças respiratórias |
| **Mecanismos pós-programa** | Mediadores | Acesso a gás de cozinha, redução de uso de biomassa, aumento de renda disponível para alimentação |
| **Dados adicionais** | Variáveis necessárias, ainda não disponíveis | Painel pré/pós; unidades contrafactuais; medida de outcome externo |

### Arestas

| Aresta | Interpretação |
|--------|---------------|
| Características prévias → Gás do Povo | Elegibilidade determina o acesso |
| Características prévias → Resultado | Confundidor direto (back-door path) |
| Gás do Povo → Resultado | **Efeito causal total** a ser estimado |
| Gás do Povo → Mecanismos → Resultado | Efeito indireto (mediado) |
| Dados adicionais ⟶ ... | Caminhos que dependem de dados ainda não coletados |

### Estratégia de identificação recomendada

Para fechar o back-door path (Características prévias → Resultado), uma futura avaliação pode usar:
- **Diferenças-em-diferenças (DiD)** com o painel município-ano antes e após a implantação do programa;
- **Propensity Score Matching (PSM)** controlando por variáveis do CadÚnico;
- **Regressão em descontinuidade (RDD)** no limiar de renda per capita (½ salário mínimo).